In [ ]:
import copy
import matplotlib.pyplot as plt
import torch
import gymnasium as gym
import torch.nn as nn
import torch.nn.functional as F
import rl_mind.envs
from dataclasses import dataclass, replace
from scipy import stats
from torch import Tensor
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from rl_mind.core import Action, Actor
from rl_mind.nn import build_mlp, soft_update
from rl_mind.env import VecEnv
from rl_mind.data import ReplayBuffer, Transitions
from rl_mind.collectors import TransitionCollector
from rl_mind.evaluation import Evaluator
from rl_mind.notebook import run_directory, setup_tensorboard, silence_known_warnings


In [ ]:
# pygame and tensorboard still use the deprecated `pkg_resources` API
silence_known_warnings()

In [ ]:
def add_layernorm(model: nn.Module):
    layers = list(model.children())
    linear_indices = [i for i, layer in enumerate(layers)if isinstance(layer, nn.Linear)]
    last_linear_index = linear_indices[-1]
    new_layers = []
    for i, layer in enumerate(layers):
        new_layers.append(layer)
        if isinstance(layer, nn.Linear) and i != last_linear_index:
            new_layers.append(nn.LayerNorm(layer.out_features))
    return nn.Sequential(*new_layers)

In [ ]:
@dataclass(frozen=True)
class DDPGConfig:
    env_name: str = "LunarLander-v3"
    seed: int = 1

    use_layernorm: bool = False

    max_steps: int = 30_000
    n_envs: int = 1
    steps_per_update: int = 1
    learning_starts: int = 1_000

    buffer_size: int = 200_000
    batch_size: int = 64

    gamma: float = 0.98
    tau: float = 0.05
    action_noise: float = 0.1

    actor_hidden: tuple[int, ...] = (64, 64)
    critic_hidden: tuple[int, ...] = (64, 64)
    lr_actor: float = 1e-3
    lr_critic: float = 1e-3

    eval_interval: int = 2_000
    n_eval_envs: int = 10

In [ ]:
class ContinuousQNetwork(nn.Module):
    """The Q-network $Q(s, a)$ for continuous actions"""

    def __init__(self, obs_dim: int, hidden: tuple[int, ...], action_dim: int, use_layernorm: bool = False):
        super().__init__()
        self.model = build_mlp([obs_dim + action_dim, *hidden, 1])
        if use_layernorm:
            self.model = add_layernorm(self.model)

    def forward(self, obs: Tensor, action: Tensor) -> Tensor:
        """Compute $Q(s, a)$ for a batch: `[B, obs_dim] x [B, action_dim] -> [B]`"""
        return self.model(torch.cat([obs, action], dim=1)).squeeze(-1)

In [ ]:
class ContinuousDeterministicActor(Actor[Action]):
    def __init__(self, obs_dim: int, hidden: tuple[int, ...], action_dim: int, use_layernorm: bool = False):
        """Creates a new actor with continuous actions in `action_dim` dimensions.

        :param obs_dim: The dimension of the observation space
        :param hidden: A tuple of integers specifying the sizes of the hidden layers
        :param action_dim: The dimension of the action space
        """
        super().__init__()
        self.model = build_mlp([obs_dim, *hidden, action_dim], output_activation=nn.Tanh())
        if use_layernorm:
            self.model = add_layernorm(self.model)

    def forward(self, obs: Tensor) -> Action:
        return Action(value=self.model(obs))

In [ ]:
class GaussianNoise(Actor[Action]):
    """Adds Gaussian noise to the actions of another actor (at training time)"""

    def __init__(self, actor: Actor[Action], sigma: float):
        super().__init__()
        self.actor = actor
        self.sigma = sigma

    def forward(self, obs: Tensor) -> Action:
        action = self.actor(obs).value
        noisy_action = action + self.sigma * torch.randn_like(action)
        noisy_action = noisy_action.clamp(-1.0, 1.0)
        return Action(value=noisy_action)

    def act(self, obs: Tensor) -> Tensor:
        return self.actor.act(obs)

In [ ]:
def compute_critic_loss(
    gamma: float, batch: Transitions[Action], q_values: Tensor, next_q_values: Tensor
) -> Tensor:
    """Compute the DDPG critic loss from a batch of transitions

    :param gamma: The discount factor
    :param batch: The batch of transitions
    :param q_values: $Q(s_t, a_t)$ from the critic (shape `[B]`)
    :param next_q_values: $Q'(s_{t+1}, \\pi(s_{t+1}))$ from the target critic
        (shape `[B]`)
    :return: The critic loss (a scalar)
    """
    # Compute the target (do not bootstrap when `batch.terminated`), then the MSE loss
    target = batch.reward + gamma * (~batch.terminated) * next_q_values
    return F.mse_loss(q_values, target)


In [ ]:
def compute_actor_loss(q_values: Tensor) -> Tensor:
    """Return the actor loss given $Q(s_t, \\pi(s_t))$ (shape `[B]`)"""
    # Compute the actor loss
    return -q_values.mean()


# TD3

In [ ]:
@dataclass(frozen=True)
class TD3Config(DDPGConfig):
    #: Number of critic updates between two policy updates
    policy_delay: int = 2
    #: Std of the noise added to the target policy actions
    target_noise: float = 0.2
    #: Clipping of the target policy noise
    target_noise_clip: float = 0.5

In [ ]:
def run_td3(cfg: TD3Config) -> Evaluator:
    torch.manual_seed(cfg.seed)
    env = VecEnv(
        cfg.env_name, cfg.n_envs,
        seed=cfg.seed, continuous=True
    )

    actor = ContinuousDeterministicActor(
        env.observation_dim, cfg.actor_hidden, env.action_dim,
        use_layernorm=cfg.use_layernorm
    )
    critic_1 = ContinuousQNetwork(
        env.observation_dim, cfg.critic_hidden, env.action_dim,
        use_layernorm=cfg.use_layernorm
    )
    critic_2 = ContinuousQNetwork(
        env.observation_dim, cfg.critic_hidden, env.action_dim,
        use_layernorm=cfg.use_layernorm
    )

    target_actor = copy.deepcopy(actor)
    target_critic_1 = copy.deepcopy(critic_1)
    target_critic_2 = copy.deepcopy(critic_2)

    actor_optimizer = torch.optim.Adam(
        actor.parameters(), lr=cfg.lr_actor
    )
    critic_optimizer = torch.optim.Adam(
        list(critic_1.parameters()) + list(critic_2.parameters()),
        lr=cfg.lr_critic
    )

    collector = TransitionCollector(
        env, GaussianNoise(actor, cfg.action_noise)
    )
    buffer = ReplayBuffer(cfg.buffer_size)

    ln_name = "LN" if cfg.use_layernorm else "noLN"
    run_dir = run_directory(
        f"td3-{cfg.env_name}-{ln_name}-S{cfg.seed}"
    )

    evaluator = Evaluator(
        VecEnv(
            cfg.env_name, cfg.n_eval_envs,
            seed=cfg.seed + 100, continuous=True
        ),
        every=cfg.eval_interval,
        run_dir=run_dir,
        writer=SummaryWriter(run_dir),
    )

    updates = 0
    pbar = tqdm(total=cfg.max_steps)

    while collector.steps < cfg.max_steps:
        buffer.add(collector.collect(cfg.steps_per_update))
        pbar.update(collector.steps - pbar.n)

        if len(buffer) < cfg.learning_starts:
            continue

        batch = buffer.sample(cfg.batch_size)

        q_values_1 = critic_1(batch.obs, batch.action.value)
        q_values_2 = critic_2(batch.obs, batch.action.value)

        # Target policy smoothing and minimum target Q-value
        with torch.no_grad():
            next_actions = target_actor(batch.next_obs).value

            noise = torch.randn_like(next_actions) * cfg.target_noise
            noise = noise.clamp(
                -cfg.target_noise_clip, cfg.target_noise_clip
            )
            next_actions = (next_actions + noise).clamp(-1.0, 1.0)

            next_q_values = torch.minimum(
                target_critic_1(batch.next_obs, next_actions),
                target_critic_2(batch.next_obs, next_actions),
            )

        critic_loss_1 = compute_critic_loss(
            cfg.gamma, batch, q_values_1, next_q_values
        )
        critic_loss_2 = compute_critic_loss(
            cfg.gamma, batch, q_values_2, next_q_values
        )
        critic_loss = critic_loss_1 + critic_loss_2

        critic_optimizer.zero_grad()
        critic_loss.backward()
        critic_optimizer.step()

        evaluator.writer.add_scalar(
            "loss/critic", critic_loss.item(), collector.steps
        )

        updates += 1

        # Delayed actor and target-network updates
        if updates % cfg.policy_delay == 0:
            actions = actor(batch.obs).value
            q_values = critic_1(batch.obs, actions)
            actor_loss = compute_actor_loss(q_values)

            actor_optimizer.zero_grad()
            actor_loss.backward()
            actor_optimizer.step()

            evaluator.writer.add_scalar(
                "loss/actor", actor_loss.item(), collector.steps
            )

            soft_update(actor, target_actor, cfg.tau)
            soft_update(critic_1, target_critic_1, cfg.tau)
            soft_update(critic_2, target_critic_2, cfg.tau)

        if result := evaluator.run_if_needed(collector.steps, actor):
            pbar.set_description(
                f"eval={result.mean:7.1f} "
                f"best={evaluator.best_reward:7.1f}"
            )

            # Monte Carlo comparison with the current actor and critic 1
            mc_env = gym.make(cfg.env_name, continuous=True)

            try:
                obs, _ = mc_env.reset(seed=cfg.seed + 1000)

                mc_return = 0.0
                discount = 1.0
                q_pred = None
                done = False

                while not done:
                    state = torch.tensor(
                        obs, dtype=torch.float32
                    ).unsqueeze(0)

                    with torch.no_grad():
                        # Deterministic action: no exploration noise
                        action = actor(state).value

                        # Predict Q only for the initial state-action pair
                        if q_pred is None:
                            q_pred = critic_1(state, action).item()

                    obs, reward, terminated, truncated, _ = mc_env.step(
                        action.squeeze(0).cpu().numpy()
                    )

                    mc_return += discount * reward
                    discount *= cfg.gamma
                    done = terminated or truncated

            finally:
                mc_env.close()

            bias = q_pred - mc_return

            evaluator.writer.add_scalar(
                "Q_vs_MC/Q_value", q_pred, collector.steps
            )
            evaluator.writer.add_scalar(
                "Q_vs_MC/Monte_Carlo", mc_return, collector.steps
            )
            evaluator.writer.add_scalar(
                "Q_vs_MC/Bias", bias, collector.steps
            )
            evaluator.writer.flush()

            tqdm.write(
                f"Step {collector.steps} | "
                f"Q1={q_pred:.2f} | "
                f"MC={mc_return:.2f} | "
                f"Bias={bias:.2f}"
            )

    evaluator.writer.flush()
    pbar.close()
    return evaluator

In [ ]:
setup_tensorboard()

In [ ]:
# Experiment 1
for i in range(5):
    td3_config = TD3Config(
      seed=i,
      use_layernorm=False,
    )
    td3_no_ln = run_td3(td3_config)

In [ ]:
for i in range(5):
    td3_config = TD3Config(
      seed=i,
      use_layernorm=True,
    )
    td3_no_ln = run_td3(td3_config)